# 日期时间与时间差

学习目标：用明确的时间单位保存日期与时间差，计算间隔，处理 NaT，并判断单位转换、时间范围和工作日规则。

前置知识：数组 dtype、类型转换、索引、日期时间的基本概念。

运行环境：Python 3.12、NumPy 2.5；示例显式指定时间单位，溢出行为按 NumPy 2.5 检查。

环境准备：[环境配置与运行](README.md)。

工作目录：本 Notebook 所在目录；重启内核后从上到下运行。

本章使用无时区的模拟记录，后续单元沿用首次导入的 np。

## 1 计算两次记录的间隔

datetime64 表示日期或时间点，timedelta64 表示带单位的时间差。两个时间点相减得到时间差，时间点加上时间差得到新的时间点。

下面两个输入都是秒精度。用时间差除以“一秒”可得到以秒计的数值，而不是把没有单位的整数直接解释成秒。

In [1]:
import numpy as np

started = np.datetime64("2026-09-20T10:00:00", "s")
finished = np.datetime64("2026-09-20T10:02:30", "s")
elapsed = finished - started

print(elapsed, elapsed.dtype)  # 150 seconds，timedelta64[s]。
print(elapsed / np.timedelta64(1, "s"))  # 150.0，以秒表示。
print(started + np.timedelta64(5, "m"))  # 2026-09-20T10:05:00。

150 seconds timedelta64[s]
150.0
2026-09-20T10:05:00


## 2 单位是类型的一部分

datetime64 与 timedelta64 都把单位放在 dtype 中。下面是常用单位，大小写有区别，M 是月，m 是分钟。

| 单位 | 中文名称／含义 |
| --- | --- |
| Y | 年 |
| M | 月 |
| W | 周 |
| D | 日 |
| h | 小时 |
| m | 分钟 |
| s | 秒 |
| ms | 毫秒 |
| us | 微秒 |
| ns | 纳秒 |

datetime64[M] 的值 2026-09 表示月精度的时间点；转换到日精度对应该月第一天。它与“一月有多少天”的持续时间问题不同。

In [2]:
month = np.datetime64("2026-09", "M")
day = month.astype("datetime64[D]")
minute = np.datetime64("2026-09-20T10:30", "m")

print(month, month.dtype)  # 2026-09 datetime64[M]。
print(day, day.dtype)  # 2026-09-01 datetime64[D]。
print(minute, minute.dtype)  # 2026-09-20T10:30 datetime64[m]。
print(np.timedelta64(1, "D") / np.timedelta64(1, "h"))  # 24.0，本类型按一天 24 小时处理。

2026-09 datetime64[M]
2026-09-01 datetime64[D]
2026-09-20T10:30 datetime64[m]
24.0


## 3 日期数组与批量运算

创建数组时显式指定 dtype，可以让整组输入采用相同单位。下面三次模拟任务的起始日期各加两天，行位置仍与原记录对应。

arange() 可创建日期范围，结束日期不包含在内。不要通过当前时间生成基础示例，以免每次运行输入都变化。

In [3]:
dates = np.array(["2026-09-18", "2026-09-20", "2026-09-23"], dtype="datetime64[D]")
due_dates = dates + np.timedelta64(2, "D")

print(due_dates)  # 2026-09-20、2026-09-22、2026-09-25。
print(due_dates.shape, due_dates.dtype)  # (3,) datetime64[D]。
print(dates[1:] - dates[:-1])  # [2 3]，dtype 为 timedelta64[D]。
print(np.arange("2026-09-18", "2026-09-22", dtype="datetime64[D]"))
# 18、19、20、21 日，不包括 22 日。

['2026-09-20' '2026-09-22' '2026-09-25']
(3,) datetime64[D]
[2 3]
['2026-09-18' '2026-09-19' '2026-09-20' '2026-09-21']


## 4 转换精度

从秒转换到分钟会丢失不足整分钟的秒数；再转回秒只能补出整分钟，不能恢复原时间。更细的单位也不自动增加原始记录的测量精度。

下面输入明确到秒，转换后观察信息变化。对本例正日期，分钟精度结果落在该分钟起点。

In [4]:
timestamps = np.array(["2026-09-20T10:30:15", "2026-09-20T10:31:45"], dtype="datetime64[s]")
minutes = timestamps.astype("datetime64[m]")
restored = minutes.astype("datetime64[s]")
milliseconds = timestamps.astype("datetime64[ms]")

print(minutes)  # 10:30 与 10:31，秒信息被去除。
print(restored)  # 秒部分都为 :00。
print(np.array_equal(restored, timestamps))  # False，原秒数不能恢复。
print(milliseconds)  # 保留原秒数，毫秒部分为 .000。
print(milliseconds.dtype)  # datetime64[ms]，不表示输入有真实毫秒测量。

['2026-09-20T10:30' '2026-09-20T10:31']
['2026-09-20T10:30:00' '2026-09-20T10:31:00']
False
['2026-09-20T10:30:15.000' '2026-09-20T10:31:45.000']
datetime64[ms]


## 5 时间范围与溢出

日期时间底层使用 64 位整数和单位。单位越细，可表示的时间范围越窄；纳秒单位覆盖的范围大约位于 1678—2262 年间，不能用它表示任意年份。

下面日精度可以表示 2500 年，转换为纳秒却超出范围。本环境的单位转换抛出 OverflowError，这属于既有的转换范围检查。NumPy 2.5 的新增变化是日期时间加减与整数乘法溢出也改为报错，不再沿用这些算术操作的旧版静默回绕行为。

In [5]:
future = np.datetime64("2500-01-01", "D")
print(future, future.dtype)  # 2500-01-01 datetime64[D]。
try:
    future.astype("datetime64[ns]")
except OverflowError as error:
    print(type(error).__name__)  # OverflowError，目标单位范围不足。
else:
    raise AssertionError("该日期无法以纳秒单位表示")

2500-01-01 datetime64[D]
OverflowError


## 6 缺失时间 NaT

NaT 表示缺失时间，可用于 datetime64 或 timedelta64。用 isnat() 识别，不能用 == NaT，因为 NaT 与自身比较也不相等。参与日期运算时，缺失会继续传播。

NumPy 2.5 开始弃用无明确单位的部分日期时间写法，本章的 NaT 同样指定单位。统计时间差前，应先按任务规则筛选有效位置。

In [6]:
dates = np.array(["2026-09-20", "NaT", "2026-09-23"], dtype="datetime64[D]")
missing = np.isnat(dates)
elapsed = dates - np.datetime64("2026-09-20", "D")

print(missing)  # [False True False]。
print(dates == np.datetime64("NaT", "D"))  # 全为 False，不能用相等比较找缺失。
print(elapsed)  # [0 NaT 3]，单位为天。
print(dates + np.timedelta64(1, "D"))  # 有效日期加一天，中间仍为 NaT。
print(elapsed[~missing] / np.timedelta64(1, "D"))  # [0. 3.]，先排除缺失再换成数值。

[False  True False]
[False False False]
[    0 'NaT'     3]
['2026-09-21'        'NaT' '2026-09-24']
[0. 3.]


## 7 月和年的持续时间

一年或一月的实际天数取决于所处日期。timedelta64 的 Y、M 不能按普通安全规则换成固定天数；unsafe 转换使用 400 年周期的平均长度，不代表某个具体月份的日数。

要得到某个月实际有多少天，可以先建立该月和下一月的起点，统一为日精度后相减。不要把“一个月”直接替换成固定 30 天。

In [7]:
one_month = np.timedelta64(1, "M")
try:
    one_month.astype("timedelta64[D]", casting="same_kind")
except TypeError as error:
    print(type(error).__name__)  # TypeError，非线性的月份长度不能直接作为固定天数。
else:
    raise AssertionError("月份时间差不应按 same_kind 转为天")

starts = np.array(["2024-02", "2025-02"], dtype="datetime64[M]")
following = starts + np.timedelta64(1, "M")
days = following.astype("datetime64[D]") - starts.astype("datetime64[D]")
print(days)  # [29 28]，分别由两个实际月份边界确定。
print(np.timedelta64(1, "Y").astype("timedelta64[M]"))  # 12 months，年与月之间可换算。

TypeError
[29 28]
12 months


## 8 时间语义的边界

datetime64 不保存时区，也没有完整的闰秒支持，每天按 86400 秒处理。带时区偏移字符串的兼容解析已经弃用，不能把它当作可靠的时区存储方案。

下面两组字符串只有表盘时间。它们是否来自相同时区、是否代表同一时刻，必须由数据来源另行约定；单靠 dtype 无法判断。这里假定两者采用同一个固定时间基准，才计算差值。

In [8]:
first = np.datetime64("2026-09-20T10:00:00", "s")
second = np.datetime64("2026-09-20T11:00:00", "s")

print(second - first)  # 3600 seconds，仅在本例共同时间基准约定下解释。
print(first.dtype)  # datetime64[s]，没有时区字段。
# 跨时区、夏令时和闰秒任务应先用能表达相应语义的工具处理，再决定是否转换。

3600 seconds
datetime64[s]


## 9 选学：工作日规则

### 9.1 判断、偏移与计数

工作日函数处理日精度日期。默认把周一到周五作为有效日；is_busday() 判断日期，busday_offset() 先按 roll 调整无效日期，再偏移指定工作日数。

busday_count() 统计区间中的有效日，不包括结束日期。这个“工作日”由日历规则定义，不自动包含任何地区的法定假日。

In [9]:
dates = np.array(["2026-09-18", "2026-09-19", "2026-09-21"], dtype="datetime64[D]")
print(np.is_busday(dates))  # [True False True]：周五、周六、周一。
print(np.busday_offset("2026-09-18", 1))  # 2026-09-21，周五后一个工作日。
print(np.busday_offset("2026-09-19", 0, roll="forward"))  # 2026-09-21，周六先向前调整。
print(np.busday_count("2026-09-18", "2026-09-22"))  # 2，统计周五和周一，不包含周二。

[ True False  True]
2026-09-21
2026-09-21
2


### 9.2 自定义工作日历

busdaycalendar() 用 weekmask 指定每周有效日，holidays 再排除特定日期。下面把 2026-09-21 设为模拟停工日，不表示它是真实法定假日。

需要共用相同规则时，把日历对象传给各个工作日函数，避免分别维护多份日期规则。

In [10]:
calendar = np.busdaycalendar(weekmask="Mon Tue Wed Thu Fri", holidays=["2026-09-21"])
print(np.is_busday(np.array(["2026-09-21", "2026-09-22"], dtype="datetime64[D]"), busdaycal=calendar))
# [False True]，模拟停工日被排除。
print(np.busday_offset("2026-09-18", 1, busdaycal=calendar))  # 2026-09-22。
print(np.busday_count("2026-09-18", "2026-09-22", busdaycal=calendar))  # 1，只计周五。

[False  True]
2026-09-22
1


## 本章小结

（1）datetime64 表示时间点，timedelta64 表示时间差；单位是 dtype 的组成部分。

（2）降低单位精度会丢失细节，提高单位精度又受表示范围限制。

（3）NaT 用 isnat 识别；月、年的时间差不能随意换成固定天数。

（4）时区和闰秒不由 datetime64 完整表达；工作日函数只执行显式提供的日历规则。

## 练习

（1）计算三次任务的耗时，保留有效记录，输出秒数。说明为什么需要先处理 NaT。

In [11]:
starts = np.array(["2026-09-20T10:00:00", "2026-09-20T11:00:00", "NaT"], dtype="datetime64[s]")
ends = np.array(["2026-09-20T10:01:00", "2026-09-20T11:02:30", "2026-09-20T12:00:00"], dtype="datetime64[s]")

# 在此计算有效掩码与时间差；检查：有效秒数为 [60, 150]。
# 有效性需要同时考虑起点和终点，不用 == NaT 判断。

（2）任务甲只记录 2500 年的某一天，任务乙记录当前年代的毫秒时间。分别选择 dtype 并解释理由；为什么不能统一选最细的 ns 就认为一定更好？

In [12]:
future_text = "2500-01-01"
precise_text = "2026-09-20T10:00:00.125"

# 在此分别选择单位，打印值和 dtype。
# 检查：甲保留日期，乙保留 .125 秒；说明精度与范围的取舍。

（3）先预测以下转换保留了哪些信息，再运行核对。解释为什么回到秒精度仍无法恢复原值。

In [13]:
original = np.datetime64("2026-09-20T10:30:45", "s")
coarse = original.astype("datetime64[m]")
restored = coarse.astype("datetime64[s]")

# 先写预测，再核对。
print(coarse)
print(restored)
print(original == restored)

2026-09-20T10:30
2026-09-20T10:30:00
False


（4）选学：工作日为周一至周五，模拟停工日为 2026-09-21。求周五 2026-09-18 之后两个工作日的日期，并统计到该日期之前的有效日数。说明结束日期是否计入。

In [14]:
calendar = np.busdaycalendar(weekmask="Mon Tue Wed Thu Fri", holidays=["2026-09-21"])

# 在此偏移和计数；日期应为 2026-09-23。
# 区间从 18 日开始、不含 23 日，计数应为 2；不要混淆偏移起点和区间端点。

## 参考与引用来源

| 网站 | 本章参考内容与定位 |
| --- | --- |
| NumPy 官方文档（NumPy 2.5） | [Datetimes and timedeltas](https://numpy.org/doc/2.5/reference/arrays.datetime.html) 的 conventions and assumptions、Basic datetimes、arithmetic、Datetime units、Business day functionality 与 generic units 迁移：类型、单位、范围、NaT、月年换算、时区及闰秒限制；[2.5 发布说明](https://numpy.org/doc/2.5/release/2.5.0-notes.html#datetime64-timedelta64-arithmetic-raises-on-overflow) 的溢出报错与既有单位转换检查；[isnat](https://numpy.org/doc/2.5/reference/generated/numpy.isnat.html) 的缺失检测；[busdaycalendar](https://numpy.org/doc/2.5/reference/generated/numpy.busdaycalendar.html) 的 weekmask、holidays；[is_busday](https://numpy.org/doc/2.5/reference/generated/numpy.is_busday.html)、[busday_offset](https://numpy.org/doc/2.5/reference/generated/numpy.busday_offset.html)、[busday_count](https://numpy.org/doc/2.5/reference/generated/numpy.busday_count.html) 的日精度、roll 与不包含结束日期规则。 |